# 01. Data Preprocessing

This notebook loads the raw RACE dataset and converts it into the **verification format**.
In verification mode, each multiple-choice question is expanded into 4 separate rows (one for each option).
The objective changes from "which option is correct?" to "is this specific option correct? (0 or 1)".

## Key Steps:
1. Load train/val/test CSVs.
2. Text cleaning (lowercase, whitespace).
3. Expand into verification rows.
4. Assign `sample_id` (`id__original_row_index`) to safely group options belonging to the same question later.
5. Save the processed splits to `data/processed/`.


In [1]:
import os
import pandas as pd
import numpy as np
import re
from pathlib import Path

# Setup paths
RAW_DIR = Path("../data/raw")
PROCESSED_DIR = Path("../data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("Directories ready.")


Directories ready.


In [2]:
def clean_text(text):
    """Basic lowercase and whitespace normalization."""
    if pd.isna(text):
        return ""
    text = str(text).lower()
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

def expand_to_verification(df):
    """
    Expands a dataframe of MCQs into verification rows.
    Each row in the output represents a single (article, question, option) triple.
    """
    rows = []
    for idx, row in df.iterrows():
        base_id = str(row.get('id', 'unk'))
        # Create a unique sample_id for the MCQ group
        sample_id = f"{base_id}__{idx}"
        
        article = clean_text(row['article'])
        question = clean_text(row['question'])
        correct_ans = str(row['answer']).strip().upper()
        
        # Expand into 4 rows
        for option_letter in ['A', 'B', 'C', 'D']:
            if option_letter not in row:
                continue
            
            option_text = clean_text(row[option_letter])
            is_correct = 1 if option_letter == correct_ans else 0
            
            rows.append({
                'sample_id': sample_id,
                'article': article,
                'question': question,
                'option_text': option_text,
                'label': is_correct,
                'option_letter': option_letter
            })
            
    return pd.DataFrame(rows)


In [3]:
print("Processing TRAIN set...")
train_df = pd.read_csv(RAW_DIR / "train.csv")
train_ver = expand_to_verification(train_df)
train_ver.to_csv(PROCESSED_DIR / "train_verification.csv", index=False)
print(f"  Raw rows: {len(train_df)}")
print(f"  Verification rows: {len(train_ver)}")

print("\nProcessing VAL set...")
val_df = pd.read_csv(RAW_DIR / "val.csv")
val_ver = expand_to_verification(val_df)
val_ver.to_csv(PROCESSED_DIR / "val_verification.csv", index=False)
print(f"  Raw rows: {len(val_df)}")
print(f"  Verification rows: {len(val_ver)}")

print("\nProcessing TEST set...")
# If you don't have test.csv, we can handle it gracefully
if (RAW_DIR / "test.csv").exists():
    test_df = pd.read_csv(RAW_DIR / "test.csv")
    test_ver = expand_to_verification(test_df)
    test_ver.to_csv(PROCESSED_DIR / "test_verification.csv", index=False)
    print(f"  Raw rows: {len(test_df)}")
    print(f"  Verification rows: {len(test_ver)}")
else:
    print("  test.csv not found, skipping.")

print("\nPreprocessing complete! Data saved to data/processed/")


Processing TRAIN set...
  Raw rows: 70258
  Verification rows: 281032

Processing VAL set...
  Raw rows: 8859
  Verification rows: 35436

Processing TEST set...
  Raw rows: 8735
  Verification rows: 34940

Preprocessing complete! Data saved to data/processed/


In [4]:
# Quick sanity check on the output
display(train_ver.head(8))
print(f"Positive class ratio: {train_ver['label'].mean():.4f}")


,sample_id,article,question,option_text,label,option_letter
0,middle7348.txt__0,in the summer between my first year and second...,before the writer came to the high school summ...,instructor,0,A
1,middle7348.txt__0,in the summer between my first year and second...,before the writer came to the high school summ...,camper,0,B
2,middle7348.txt__0,in the summer between my first year and second...,before the writer came to the high school summ...,student,1,C
3,middle7348.txt__0,in the summer between my first year and second...,before the writer came to the high school summ...,reporter,0,D
4,middle7348.txt__1,in the summer between my first year and second...,how many times did the writer invite the boy t...,once,0,A
5,middle7348.txt__1,in the summer between my first year and second...,how many times did the writer invite the boy t...,twice,1,B
6,middle7348.txt__1,in the summer between my first year and second...,how many times did the writer invite the boy t...,three times,0,C
7,middle7348.txt__1,in the summer between my first year and second...,how many times did the writer invite the boy t...,many times,0,D


Positive class ratio: 0.2500
